In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-03-16 19:56:23 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-03-16 19:56:23 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=7, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-03-16 19:56:23 __main__ INFO     transformers.__version__='5.3.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-03-16 19:56:27 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-03-16 19:56:27 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-16 19:56:27 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f908cfcc0b0>
2026-03-16 19:56:27 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f8fdf09c850> server_hostname='huggingface.co' timeout=10
2026-03-16 19:56:27 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f8fdeafa720>
2026-03-16 19:56:27 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-16 19:56:27 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-16 19:56:27 http

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-03-16 19:56:50 httpcore.connection DEBUG    close.started
2026-03-16 19:56:50 httpcore.connection DEBUG    close.complete
2026-03-16 19:56:50 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-16 19:56:50 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f8fdd2d4a10>
2026-03-16 19:56:50 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f8fdf09c850> server_hostname='huggingface.co' timeout=10
2026-03-16 19:56:51 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f8fdd2d7920>
2026-03-16 19:56:51 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-16 19:56:51 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-16 19:56:51 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

## Debug

In [58]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.predicate_data import MultiPredicateDataset

PREDICATE_IDX = 2

eval_dataset = MultiPredicateDataset.load_dataset(augment=False)
target_predicate = eval_dataset[PREDICATE_IDX]

orig_samples = [(i, s) for i, s in enumerate(target_predicate.samples) if s.is_original]
orig_indices = [i for i, _ in orig_samples]

rich_print(
    f"[bold]{target_predicate.name}[/bold]  "
    f"[green]{len(orig_samples)} originals[/green] + "
    f"[yellow]{len(target_predicate.samples) - len(orig_samples)} augmented[/yellow] "
    f"= {len(target_predicate.samples)} total samples (augmented in-memory)"
)
rich_print(target_predicate.p_plus)

2026-03-16 18:28:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'has_unique_elements' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/has_unique_elements.json


2026-03-16 18:28:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_anagram' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_anagram.json
2026-03-16 18:28:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_palindrome' with 5 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json
2026-03-16 18:28:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_perfect_square' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_perfect_square.json
2026-03-16 18:28:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_power_of_two' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_power_of_two.json
2026-03-16 18:28:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_sorted' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json
2026-03-16 18:28:40 src.goals.predicate_data INFO   

is_palindrome  5 originals + 0 augmented = 5 total samples (augmented in-memory)

PredicateSpec(
    description='check whether s is a palindrome (s == s[::-1])',
    short_description='{var} is a palindrome',
    anchor_patterns=[
        '//2\\]',
        '::\\-1\\]',
        '==\\ \\w+',
        '==\\ \\w+\\ ==',
        '==\\ \\w+\\.lower\\(\\)\\[::\\-1\\]',
        '==\\ \\w+\\[',
        '==\\ \\w+\\[::\\-1\\]',
        '==\\w+',
        '>>\\ 1\\]',
        '\\[::\\-1\\]',
        'enumerate\\(\\w+\\)',
        'lower\\(\\)\\[::\\-1\\]',
        'range\\(\\ len\\(\\ \\w+\\ \\)\\ //\\ 2\\ \\)',
        'reversed',
        'reversed\\(\\w+\\)',
        '\\w+\\ !=\\ \\w+',
        '\\w+\\ !=\\ \\w+\\[',
        '\\w+\\ !=\\ \\w+\\[::\\-1\\]',
        '\\w+\\ ==',
        '\\w+\\ ==\\ \\w+',
        '\\w+\\ ==\\ \\w+\\[',
        '\\w+\\ ==\\ \\w+\\[::\\-1\\]',
        '\\w+\\.lower\\(\\)\\ ==',
        '\\w+\\.lower\\(\\)\\[::\\-1\\]',
        '\\w+\\[\\-\\(',
        '\\w+\\[\\-1::\\-1\\]',
        '\\w+\\[\\-i\\-1\\]',
        '\\w+\\[\\-len',
        '\\w+\\[:\\(len',
        '\\w+\\[::\\-',
        '\\w+\\[::\\-\\ 1\\]',
        '\\w+\\[::\\-1\\]',
        '\\w+\\[i\\]\\ !=',
        '\\w+\\[i\\]\\ !=\\ \\w+\\[len\\(\\w+\\)\\ \\-\\ 1\\ \\-\\ i\\]',
        '\\w+\\[i\\]\\ !=\\ \\w+\\[\\~i\\]',
        '\\w+\\[i\\]\\ ==\\ \\w+\\[\\-i\\-1\\]',
        'upper\\(\\)==\\w+\\[::\\-1\\]',
        'upper\\(\\)\\[::\\-1\\]',
        '\\~i'
    ]
)

In [17]:
len(target_predicate.samples)

45

In [21]:
sample = target_predicate.samples[10]

In [22]:
rich_print(sample)

MultiPredicateSample(
    instruction='Write a Python function that returns True iff the string has at least 2 characters and is a 
palindrome.\n\n',
    code='def has_valid_palindrome(s: str) -> bool:\n    """Return True iff s has at least 2 characters and is a 
palindrome."""\n    return ',
    unit_tests=[
        ('aba', True),
        ('ab', False),
        ('a', False),
        ('', False),
        ('aa', True),
        ('abcba', True),
        ('abc', False),
        ('racecar', True),
        ('hello', False)
    ],
    prefills={
        'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
        'aligned_p_minus': ['len(', 'len(s)'],
        'adversarial': ['not (', 'False if', 'True if not'],
        'inconsequential': ['']
    },
    function_name='has_valid_palindrome',
    augmentation=[
        {'type': 'function_rename', 'old_name': 'is_valid_palindrome', 'new_name': 'has_valid_palindrome'}
    ],
    p_minus=PredicateSpec(
        description='check whether len(s) >= 2',
        short_description='{var} has at least 2 characters',
        anchor_patterns=[
            "!=\\ '\\ '",
            '!=\\ \\-1',
            "'\\ '\\ in\\ \\w+",
            '\\.count\\(',
            '>\\ 0',
            'and\\ not',
            'any\\(',
            "c\\ ==\\ '\\ '",
            "count\\('\\ '\\)",
            "find\\('\\ '\\)",
            'isspace',
            'isspace\\(\\)',
            'len\\(',
            'len\\(\\w+\\)',
            'not\\ "\\ "\\ in\\ \\w+',
            'not\\ in\\ \\w+',
            '\\w+\\ and',
            'sum\\(1\\ for\\ c\\ in\\ \\w+'
        ]
    ),
    p_plus_short_description='{var} is a palindrome'
)

In [23]:
s_aug = sample.augment(
    augmentation_type="variable_rename",
    augmentation_config={"old_var": "s", "new_var": "var"},
)
rich_print(s_aug.code)
s_aug2 = s_aug.augment("swap_predicate_order", {})
rich_print(s_aug2.code)

s_aug3 = s_aug2.augment("swap_predicate_order", {})
rich_print(s_aug3.code)

def has_valid_palindrome(var: str) -> bool:
    """Return True iff var has at least 2 characters and is a palindrome."""
    return

def has_valid_palindrome(var: str) -> bool:
    """Return True iff var is a palindrome and var has at least 2 characters."""
    return

def has_valid_palindrome(var: str) -> bool:
    """Return True iff var is a palindrome and var has at least 2 characters."""
    return

## Inspect Augmented Variants

In [24]:
rich_print(target_predicate.samples[2])

MultiPredicateSample(
    instruction='Write a Python function that returns True iff the string contains no spaces and is a 
palindrome.\n\n',
    code='def is_spaceless_palindrome(s: str) -> bool:\n    """Return True iff s contains no spaces and is a 
palindrome."""\n    return ',
    unit_tests=[
        ('aba', True),
        ('abc', False),
        ('a b a', False),
        ('race car', False),
        ('racecar', True),
        ('hello', False),
        ('a', True),
        ('', True),
        ('a a', False),
        ('abcba', True)
    ],
    prefills={
        'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
        'aligned_p_minus': ["' ' not in", '" " not in'],
        'adversarial': ['not (', 'False if', 'True if not'],
        'inconsequential': ['']
    },
    function_name='is_spaceless_palindrome',
    augmentation=None,
    p_minus=PredicateSpec(
        description="check whether ' ' not in s (no spaces)",
        short_description='{var} contains no spaces',
        anchor_patterns=['alpha\\(\\)', 'isalpha']
    ),
    p_plus_short_description='{var} is a palindrome'
)

In [27]:
# Show one example of each augmentation type for the target predicate.
# Verify code stubs and prefills look correct before running MCTS.

seen_types: dict = {}
for idx, s in enumerate(target_predicate.samples):
    if s.is_original:
        continue
    atype = s.augmentation[-1].get("type", "unknown") if s.augmentation else "unknown"
    if atype not in seen_types:
        seen_types[atype] = (idx, s)

print(f"Seen augmentation types for predicate {target_predicate.name!r}: {len(seen_types)}")

for atype, (idx, s) in seen_types.items():
    rich_print(f"sample_idx={idx} | [bold cyan]augmentation type: {atype!r}[/bold cyan]  "
               f"fn=[magenta]{s.function_name!r}[/magenta]")
    rich_print(f"  augmentation : {s.augmentation}")
    rich_print(f"  code         :\n[white]{s.code}[/white]")
    if hasattr(s, 'p_minus') and s.p_minus:
        rich_print(f"  p_minus.short_description : {s.p_minus.short_description!r}")
    # Show a prefill sample to verify variable renaming propagated
    for cat, pfills in s.prefills.items():
        rich_print(f"  prefills[{cat!r}] : {pfills[:3]}")
        break

Seen augmentation types for predicate 'is_palindrome': 4


sample_idx=5 | augmentation type: 'variable_rename'  fn='is_valid_palindrome'

augmentation : [{'type': 'variable_rename', 'mapping': {'s': 'text'}}]

code         :
def is_valid_palindrome(text: str) -> bool:
    """Return True iff text has at least 2 characters and is a palindrome."""
    return 

p_minus.short_description : '{var} has at least 2 characters'

prefills['aligned_p_plus'] : ['text == text', 'text[::-1]', 'text[:']

... (11 outputs left out)

augmentation : [{'type': 'swap_predicate_order'}]

code         :
def is_valid_palindrome(s: str) -> bool:
    """Return True iff s is a palindrome and s has at least 2 characters."""
    return 

p_minus.short_description : '{var} has at least 2 characters'

prefills['aligned_p_plus'] : ['s == s', 's[::-1]', 's[:']

In [29]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

# SAMPLE_IDX: 0-based index into originals (not the full augmented list).
SAMPLE_IDX = 12
sample = target_predicate.samples[12]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

eval_results = run_evaluation(
    result=results,
    function_name=function_name,
    code_stop_tag="</code>",
)

pretty_print_eval_results(eval_results)

"""def is_valid_palindrome(s: str) -> bool:
    """Return True iff s is a palindrome and s has at least 2 characters."""
    return """

                                                             Code Completion Results                                                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                    ┃ Logic                                             ┃ Verdict ┃ Unit Tests                                   ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ aligned_p_plus: "s == s"   │ "s == s[::-1] and len(s) >= 2"                    │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ aligned_p_plus: "s[::-1]"  │ "s[::-1] == s and len(s) >= 2"                    │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ aligned_p_plus: "s[:"      │ "s[:].lower() == s[::-1].lower() and len(s) >= 2" │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ aligned_p_plus: "s =="     │ "s == s[::-1] and len(s) >= 2"                    │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ aligned_p_minus: "len("    │ "len( s) >= 2 and s == s[::-1]"                   │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ aligned_p_minus: "len(s)"  │ "len(s) >= 2 and s == s[::-1]"                    │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ adversarial: "not ("       │ "not (len(s) < 2) and s == s[::-1]"               │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ adversarial: "False if"    │ "False if len(s) < 2 else s == s[::-1]"           │ PASSED  │                                              │      │
├────────────────────────────┼───────────────────────────────────────────────────┼─────────┼──────────────────────────────────────────────┼──────┤
│ adversarial: "True if not" │ "True if not s else len(s) > 1 and s == s[::-1]"  │ FAILED  │ is_valid_palindrome(aba)=True ✓              │      │
│                            │                                                   │         │ is_valid_palindrome(ab)=False ✓              │      │
│                            │                                                   │         │ is_valid_palindrome(a)=False ✓               │      │
│                            │                                                   │         │ is_valid_palindrome()=True, expected False ✗ │      │
│                            │                                                   │         │ is_valid_palindrome(aa)=True ✓               │      │
│                            │                                                   │         │ is_valid_palindrome(abcba)=True ✓            │      │
│                            │                                                   │         │ is_valid_palindrome(abc)=False ✓             │      │
│                            │ 

Summary: 9/10 passed, 1/10 failed

In [32]:
print(prompts[0])

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python function that returns True iff the input is a palindrome and the input has at least 2 characters.<|im_end|>
<|im_start|>assistant
<code>
def is_valid_palindrome(s: str) -> bool:
    """Return True iff s is a palindrome and s has at least 2 characters."""
    return s == s


In [35]:
prefill_idx = -1
generated_code = eval_results.generations[prefill_idx]
print(generated_code)

print(f'Full Prompt:\n"{generated_code.full_prompt}"\n')
print(f'Full Output:\n"{generated_code.lm_generation}"\n')

Code(prefill='', prefill_type='inconsequential', full_prompt='<|im_start|>system\nYou are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n<|im_start|>user\nWrite a Python function that returns True iff the input is a palindrome and the input has at least 2 characters.<|im_end|>\n<|im_start|>assistant\n<code>\ndef is_valid_palindrome(s: str) -> bool:\n    """Return True iff s is a palindrome and s has at least 2 characters."""\n    return ', lm_generation=' len(s) >= 2 and s == s[::-1]', source_code='def is_valid_palindrome(s: str) -> bool:\n    """Return True iff s is a palindrome and s has at least 2 characters."""\n    return  len(s) >= 2 and s == s[::-1]', function_name='is_valid_palindrome', eval=SingleCodeEval(code='def is_valid_palindrome(s: str) -> bool:\n    """Return True iff s is a palindrome and s has at least 2 characters."""\n    return 

In [36]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[prefill_idx]
print(f'Full Prompt for MCTS:\n"{full_prompt}"\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=32, top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)

root = search.run(
    n_iterations=1500,
    depth_first=True,
)

Full Prompt for MCTS:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python function that returns True iff the input is a palindrome and the input has at least 2 characters.<|im_end|>
<|im_start|>assistant
<code>
def is_valid_palindrome(s: str) -> bool:
    """Return True iff s is a palindrome and s has at least 2 characters."""
    return "

2026-03-16 18:17:11 src.goals.sampling DEBUG    MCTS iteration 50/1500, total nodes: 46
2026-03-16 18:17:11 src.goals.sampling DEBUG    MCTS iteration 100/1500, total nodes: 46
2026-03-16 18:17:11 src.goals.sampling DEBUG    MCTS iteration 150/1500, total nodes: 46
2026-03-16 18:17:11 src.goals.sampling DEBUG    MCTS iteration 200/1500, total nodes: 46
2026-03-16 18:17:11 src.goals.sampling DEBUG    MCTS iteration 250/1500, total nodes: 46
2026-03-16 18:17:11 

In [37]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> {repr(p["text"])}'
    )

{'total_iterations': 1500, 'total_nodes': 55, 'terminal_count': 9, 'max_depth_reached': 13, 'avg_depth': 11.333333333333334, 'root_visits': 1536, 'terminal_reasons': {'stop_string:\n': 9}}
<v=1084, p=0.7063> ' len(s) >= 2 and s == s[::-1]\n'
<v=244, p=0.1629> ' len(s) > 1 and s == s[::-1]\n'
<v=146, p=0.0957> ' len(s) >= 2 and s == s[::-1]\n\n'
<v=25, p=0.0171> ' len(s) > 1 and s == s[::-1]\n\n'
<v=1, p=0.0060> '2 <= len(s) and s == s[::-1]\n'
<v=0, p=0.0010> '2 <= len(s) and s == s[::-1]\n\n'
<v=1, p=0.0009> '2 <= len(s) == len(s[::-1])\n'
<v=0, p=0.0006> '2 <= len(s) == len(s[::-1])\n\n'
<v=0, p=0.0006> '2 <= len(s) == s'
<v=0, p=0.0002> '2 <= len(s) <='
<v=0, p=0.0002> '2 <= len(s) == len(s)'
<v=0, p=0.0001> '2 &'
<v=0, p=0.0001> '2 <= len(s) == len(s[::-1])'
<v=0, p=0.0000> ' len(s)>='
<v=0, p=0.0000> ' len s'
<v=0, p=0.0000> '2 <= s'
<v=0, p=0.0000> '2 <= len(\n'


In [38]:
# Inspect results
search.print_tree(max_depth=None, show_colors=False)

[ROOT] visits=1536
    ├── ' len' (p=0.984, v=1519)
    │   ├── '(s' (p=1.000, v=1518)
    │   │   ├── ')' (p=1.000, v=1517)
    │   │   │   ├── ' >=' (p=0.816, v=1239)
    │   │   │   │   └── ' ' (p=1.000, v=1238)
    │   │   │   │       └── '2' (p=1.000, v=1237)
    │   │   │   │           └── ' and' (p=1.000, v=1236)
    │   │   │   │               └── ' s' (p=1.000, v=1235)
    │   │   │   │                   └── ' ==' (p=1.000, v=1234)
    │   │   │   │                       └── ' s' (p=1.000, v=1233)
    │   │   │   │                           └── '[::-' (p=1.000, v=1232)
    │   │   │   │                               └── '1' (p=1.000, v=1231)
    │   │   │   │                                   ├── ']\n' (p=0.879, v=1084) [TERMINAL] (' len(s) >= 2 and s == s[::-1]\n')
    │   │   │   │                                   └── ']\n\n' (p=0.119, v=146) [TERMINAL] (' len(s) >= 2 and s == s[::-1]\n\n')
    │   │   │   └── ' >' (p=0.183, v=277)
    │   │   │       └── ' ' (p=1.000, v=27

In [39]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

def is_valid_palindrome(s: str) -> bool:
    """Return True iff s is a palindrome and s has at least 2 characters."""
    return 
<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python function that returns True iff the input is a palindrome and the input has at least 2 characters.<|im_end|>
<|im_start|>assistant
<code>
def is_valid_palindrome(s: str) -> bool:
    """Return True iff s is a palindrome and s has at least 2 characters."""
    return  == n % 2)


[[PredictedToken(token=' and', prob=0.91015625, logit=20.625, token_id=323, metadata=None),
  PredictedToken(token=' else', prob=0.0189208984375, logit=16.75, token_id=770, metadata=None),
  PredictedToken(token=' for', prob=0.01470947265625, logit=16.5, token_id=369, metadata=None),
  PredictedToken(token=' ', prob=0.00787353515625, logit=15.875, token_id=220, metadata=None),
  PredictedToken(token=' if', prob=0.00738525390625, logit=15.8125, token_id=421, metadata=None)]]

In [40]:
# node = search.find_node_by_text(' == n % 2)')
# node

In [41]:
# # Debug why it's not being expanded
# if node:
#     print(search.debug_node(node))

#     # If it's not terminal, see what children would be created
#     if not node.is_terminal and node.is_leaf():
#         search.force_expand(node)

#     # Check PUCT scores at its parent to see why siblings might be selected first
#     if node.parent:
#         for child, score in search.get_puct_scores(node.parent):
#             print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

In [42]:
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors

prefix = sample.code + generated_code.prefill
clear_mcts_colors(search.root)

counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=sample.unit_tests,
    function_name=sample.function_name or "func",
    terminal_only=True
)

print(counts)

{'green': 6, 'orange': 2, 'red': 1, 'total': 9}


In [43]:
search.print_tree(max_depth=None, show_colors=True)

[ROOT] visits=1536
    ├── ' len' (p=0.984, v=1519)
    │   ├── '(s' (p=1.000, v=1518)
    │   │   ├── ')' (p=1.000, v=1517)
    │   │   │   ├── ' >=' (p=0.816, v=1239)
    │   │   │   │   └── ' ' (p=1.000, v=1238)
    │   │   │   │       └── '2' (p=1.000, v=1237)
    │   │   │   │           └── ' and' (p=1.000, v=1236)
    │   │   │   │               └── ' s' (p=1.000, v=1235)
    │   │   │   │                   └── ' ==' (p=1.000, v=1234)
    │   │   │   │                       └── ' s' (p=1.000, v=1233)
    │   │   │   │                           └── '[::-' (p=1.000, v=1232)
    │   │   │   │                               └── '1' (p=1.000, v=1231)
    │   │   │   │                                   ├── ']\n' (p=0.879, v=1084) [TERMINAL] (' len(s) >= 2 and s == s[::-1]\n')
    │   │   │   │                                   └── ']\n\n' (p=0.119, v=146) [TERMINAL] (' len(s) >= 2 and s == s[::-1]\n\n')
    │   │   │   └── ' >' (p=0.183, v=277)
    │   │   │       └── ' ' (p=1.000, v=277)
    │   │   │           └── '1' (p=1.000, v=276)
    │   │   │               └── ' and' (p=1.000, v=275)
    │   │   │                   └── ' s' (p=1.000, v=274)
    │   │   │                       └── ' ==' (p=1.000, v=273)
    │   │   │                           └── ' s' (p=1.000, v=272)
    │   │   │                               └── '[::-' (p=1.000, v=271)
    │   │   │                                   └── '1' (p=1.000, v=270)
    │   │   │                                       ├── ']\n' (p=0.906, v=244) [TERMINAL] (' len(s) > 1 and s == s[::-1]\n')
    │   │   │                                       └── ']\n\n' (p=0.095, v=25) [TERMINAL] (' len(s) > 1 and s == s[::-1]\n\n')
    │   │   └── ')>=' (p=0.000, v=0) (' len(s)>=')
    │   └── ' s' (p=0.000, v=0) (' len s')
    └── '2' (p=0.010, v=17)
        ├── ' <=' (p=0.988, v=16)
        │   ├── ' len' (p=1.000, v=15)
        │   │   ├── '(s' (p=1.000, v=14)
        │   │   │   └── ')' (p=1.000, v=13)
        │   │   │       ├── ' and' (p=0.734, v=7)
        │   │   │       │   └── ' s' (p=1.000, v=6)
        │   │   │       │       └── ' ==' (p=1.000, v=5)
        │   │   │       │           └── ' s' (p=1.000, v=4)
        │   │   │       │               └── '[::-' (p=1.000, v=3)
        │   │   │       │                   └── '1' (p=1.000, v=2)
        │   │   │       │                       ├── ']\n' (p=0.852, v=1) [TERMINAL] ('2 <= len(s) and s == s[::-1]\n')
        │   │   │       │                       └── ']\n\n' (p=0.148, v=0) [TERMINAL] ('2 <= len(s) and s == s[::-1]\n\n')
        │   │   │       ├── ' ==' (p=0.239, v=5)
        │   │   │       │   ├── ' len' (p=0.754, v=5)
        │   │   │       │   │   └── '(s' (p=0.988, v=4)
        │   │   │       │   │       ├── '[::-' (p=0.891, v=3)
        │   │   │       │   │       │   └── '1' (p=1.000, v=2)
        │   │   │       │   │       │       ├── '])\n' (p=0.566, v=1) [TERMINAL] ('2 <= len(s) == len(s[::-1])\n')
        │   │   │       │   │       │       ├── '])\n\n' (p=0.391, v=0) [TERMINAL] ('2 <= len(s) == len(s[::-1])\n\n')
        │   │   │       │   │       │       └── '])' (p=0.041, v=0) ('2 <= len(s) == len(s[::-1])')
        │   │   │       │   │       └── ')' (p=0.106, v=0) ('2 <= len(s) == len(s)')
        │   │   │       │   └── ' s' (p=0.244, v=0) ('2 <= len(s) == s')
        │   │   │       └── ' <=' (p=0.025, v=0) ('2 <= len(s) <=')
        │   │   └── '(\n' (p=0.000, v=0) [TERMINAL] ('2 <= len(\n')
        │   └── ' s' (p=0.000, v=0) ('2 <= s')
        └── ' &' (p=0.012, v=0) ('2 &')

In [44]:
search.save_tree("tree.html", show_colors=True)

2026-03-16 18:17:48 src.goals.sampling INFO     Saved tree to tree.html (HTML with colors)


## Checking the green solutions

In [45]:
# def get_green_solutions(root, prefix, tokenizer=None):                                                                                                                                                                        
#     """Return list of (full_code, eval_result) for all green leaf nodes."""                                                                                                                                                   
#     results = []                                                                                                                                                                                                              
#     def walk(node):                                                                                                                                                                                                           
#         if node.is_leaf() and node.color == "green":                                                                                                                                                                        
#             results.append((prefix + node.get_path_text(), node.eval_result))
#         for child in node.children:
#             walk(child)
#     walk(root)
#     return results

# # After evaluate_mcts_tree(root, prefix, unit_tests, function_name):
# solutions = get_green_solutions(root, prefix="")
# for code, eval_result in solutions:
#     print(code)
#     print("---")

In [46]:
def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict
    groups = defaultdict(list)
    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)
    walk(root)
    return dict(groups)

groups = get_green_solutions_by_first_token(root, prefix="")
for first_tok, codes in groups.items():
    print(f"First token: {first_tok!r}")
    for code in codes:
        print(f'"{code}"')
        # print("---")

First token: ' len'
" len(s) >= 2 and s == s[::-1]
"
" len(s) >= 2 and s == s[::-1]

"
" len(s) > 1 and s == s[::-1]
"
" len(s) > 1 and s == s[::-1]

"
First token: '2'
"2 <= len(s) and s == s[::-1]
"
"2 <= len(s) and s == s[::-1]

"


## Build Trees for the Target Predicate

In [12]:
from src.goals.predicate_data import MultiPredicateDataset

PREDICATE_IDX = 2

eval_dataset = MultiPredicateDataset.load_dataset(augment=False)
target_predicate = eval_dataset[PREDICATE_IDX]

orig_samples = [(i, s) for i, s in enumerate(target_predicate.samples) if s.is_original]
orig_indices = [i for i, _ in orig_samples]

rich_print(
    f"[bold]{target_predicate.name}[/bold]  "
    f"[green]{len(orig_samples)} originals[/green] "
    f"= {len(target_predicate.samples)} total (augmented samples are in-memory only)"
)

2026-03-16 19:57:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'has_unique_elements' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/has_unique_elements.json
2026-03-16 19:57:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_anagram' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_anagram.json
2026-03-16 19:57:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_palindrome' with 5 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json
2026-03-16 19:57:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_perfect_square' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_perfect_square.json
2026-03-16 19:57:40 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_power_of_two' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_power_of_two.json
2026-03-16 19:57:40 src.goals.pr

is_palindrome  5 originals = 5 total (augmented samples are in-memory only)

In [13]:
from src.goals.predicate_sampling import (
    MCTSRunResult,
    MCTSSearchResults_for_MultiPredicateSample,
    MCTSSearchResults_for_TargetPredicate,
)
from src.goals.predicate_data import MultiPredicateSample
from src.goals.sampling import MCTSSearch
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors
from src.goals.anchor import get_green_completions
from src.goals.functional import supports_chat_template, apply_chat_template


def build_full_prompt(sample: MultiPredicateSample, prefill: str, mt) -> str:
    if supports_chat_template(mt):
        return apply_chat_template(
            mt=mt,
            user_query=sample.instruction.strip(),
            incomplete_code=sample.code + prefill,
        )
    return sample.instruction + "<code>" + sample.code + prefill


MCTS_CONFIG = dict(
    n_iterations=1500,
    max_depth=48,
    top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=6,
    min_children=4,
    depth_first=True,
)

In [14]:
import os
from scripts.multipred_mcts import _safe_name, _p_minus_dir_name

target_predicate = eval_dataset[PREDICATE_IDX]
model_tag = mt.name.split("/")[-1]

PREDICATE_DIR = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "multipred_mcts",
    model_tag,
    target_predicate.name,
)

os.makedirs(PREDICATE_DIR, exist_ok=True)

# ── Load or run MCTS per sample ──────────────────────────────────────────────
# Only run on originals — augmented samples share the same unit tests and
# running MCTS on them would be redundant and expensive.
tp_results = MCTSSearchResults_for_TargetPredicate(
    target_predicate_name=target_predicate.name,
    model_name=mt.name,
    mcts_config=MCTS_CONFIG,
)

for sample_idx, p_neg_sample in orig_samples:  # (global_idx, sample) — originals only
    p_minus_name = _p_minus_dir_name(p_neg_sample, sample_idx)
    sample_dir = os.path.join(PREDICATE_DIR, p_minus_name)
    cache_path = os.path.join(sample_dir, "mcts.json")

    # ── Try loading from per-sample cache ────────────────────────────────
    if os.path.exists(cache_path):
        rich_print(f"[yellow]Loading cached sample {sample_idx} from {cache_path}[/yellow]")
        sample_result = MCTSSearchResults_for_MultiPredicateSample.load(cache_path)
        tp_results.sample_results.append(sample_result)
        continue

    # ── Run MCTS ─────────────────────────────────────────────────────────
    rich_print(
        f"\n[bold]Sample {sample_idx + 1}/{len(orig_samples)}[/bold]"
        f"  fn=[cyan]{p_neg_sample.function_name}[/cyan]"
        f"  p-=[italic]{p_neg_sample.p_minus.description if p_neg_sample.p_minus else '?'}[/italic]"
    )

    sample_result = MCTSSearchResults_for_MultiPredicateSample(sample=p_neg_sample)

    all_prefills = [
        (ptype, prefill)
        for ptype, prefills in p_neg_sample.prefills.items()
        for prefill in prefills
    ]

    for prefill_type, prefill in all_prefills:
        prompt = build_full_prompt(p_neg_sample, prefill, mt)
        prefix = p_neg_sample.code + prefill

        print(f"  {prefill_type} | prefill={prefill!r}")

        # 1) MCTS search
        search = MCTSSearch(
            mt=mt,
            prompt=prompt,
            max_depth=MCTS_CONFIG["max_depth"],
            top_k=MCTS_CONFIG["top_k"],
            prob_threshold=MCTS_CONFIG["prob_threshold"],
            temperature=MCTS_CONFIG["temperature"],
            c_puct=MCTS_CONFIG["c_puct"],
            min_children_up_to_depth=MCTS_CONFIG["min_children_up_to_depth"],
            min_children=MCTS_CONFIG["min_children"],
        )
        root = search.run(
            n_iterations=MCTS_CONFIG["n_iterations"],
            depth_first=MCTS_CONFIG["depth_first"],
        )

        # 2) Evaluate tree (sets .color and .eval_result on leaves)
        counts = evaluate_mcts_tree(
            root=root,
            prefix=prefix,
            unit_tests=p_neg_sample.unit_tests,
            function_name=p_neg_sample.function_name or "func",
            terminal_only=True,
        )

        # 3) Extract unique green completions
        green_completions = sorted(set(
            c.strip() for c in get_green_completions(root, prefix) if c.strip()
        ))

        run = MCTSRunResult(
            prefill=prefill,
            prefill_type=prefill_type,
            prompt=prompt,
            prefix=prefix,
            mcts_config=MCTS_CONFIG,
            mcts_stats=search.get_stats(),
            eval_counts=counts,
            green_completions=green_completions,
            tree=root.to_dict(),
        )
        sample_result.runs.append(run)
        sample_result._trees[prefix] = root

        # Save HTML tree into trees/ subdirectory
        safe_pval = _safe_name(prefill) if prefill else "empty"
        tree_dir = os.path.join(sample_dir, "trees")
        os.makedirs(tree_dir, exist_ok=True)
        html_path = os.path.join(tree_dir, f"{safe_pval}.html")
        search.save_tree(html_path, show_colors=True)

        rich_print(f"    eval: {counts} | {len(green_completions)} unique green completions")

    # Save per-sample
    sample_result.save(cache_path)
    rich_print(f"  [green]Saved → {cache_path}[/green]")
    tp_results.sample_results.append(sample_result)

# ── Summary ───────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]Summary[/bold] — [bold][cyan]{tp_results.target_predicate_name}[/cyan][/bold]  ({tp_results.model_name})")
for i, sr in enumerate(tp_results.sample_results):
    rich_print(f"sample {i} --- [bold][magenta]{sr.sample.function_name}[/magenta][bold]")
    for run in sr.runs:
        print(
            f"  [{run.prefill_type}] prefill={run.prefill!r:10s}"
            f" | {run.eval_counts} | {len(run.green_completions)} green"
        )

Loading cached sample 0 from 
/disk/u/arnab/Codes/Projects/goals/results/multipred_mcts/Qwen2.5-Coder-32B-Instruct/is_palindrome/check_whether_le
ns__2/mcts.json

2026-03-16 19:57:49 src.goals.predicate_sampling INFO     Loaded MCTSSearchResults for sample from /disk/u/arnab/Codes/Projects/goals/results/multipred_mcts/Qwen2.5-Coder-32B-Instruct/is_palindrome/check_whether_lens__2/mcts.json


Loading cached sample 1 from 
/disk/u/arnab/Codes/Projects/goals/results/multipred_mcts/Qwen2.5-Coder-32B-Instruct/is_palindrome/check_whether_s.
isalpha_all_characters_a/mcts.json

2026-03-16 19:57:49 src.goals.predicate_sampling INFO     Loaded MCTSSearchResults for sample from /disk/u/arnab/Codes/Projects/goals/results/multipred_mcts/Qwen2.5-Coder-32B-Instruct/is_palindrome/check_whether_s.isalpha_all_characters_a/mcts.json


Loading cached sample 2 from 
/disk/u/arnab/Codes/Projects/goals/results/multipred_mcts/Qwen2.5-Coder-32B-Instruct/is_palindrome/check_whether___
not_in_s_no_spaces/mcts.json

2026-03-16 19:57:50 src.goals.predicate_sampling INFO     Loaded MCTSSearchResults for sample from /disk/u/arnab/Codes/Projects/goals/results/multipred_mcts/Qwen2.5-Coder-32B-Instruct/is_palindrome/check_whether___not_in_s_no_spaces/mcts.json


... (11 outputs left out)

sample 3 --- is_upper_palindrome

  [aligned_p_plus] prefill='s == s'   | {'green': 18, 'orange': 30, 'red': 2, 'total': 50} | 18 green
  [aligned_p_plus] prefill='s[::-1]'  | {'green': 6, 'orange': 30, 'red': 6, 'total': 42} | 6 green
  [aligned_p_plus] prefill='s[:'      | {'green': 18, 'orange': 21, 'red': 0, 'total': 39} | 18 green
  [aligned_p_plus] prefill='s =='     | {'green': 16, 'orange': 21, 'red': 0, 'total': 37} | 16 green
  [aligned_p_minus] prefill='s[0].isupper()' | {'green': 2, 'orange': 15, 'red': 9, 'total': 26} | 2 green
  [aligned_p_minus] prefill='s and s[0]' | {'green': 0, 'orange': 8, 'red': 2, 'total': 10} | 0 green
  [aligned_p_minus] prefill='s[0]'     | {'green': 6, 'orange': 20, 'red': 2, 'total': 28} | 6 green
  [adversarial] prefill='not ('    | {'green': 44, 'orange': 43, 'red': 7, 'total': 94} | 44 green
  [adversarial] prefill='False if' | {'green': 36, 'orange': 4, 'red': 4, 'total': 44} | 36 green
  [adversarial] prefill='True if not' | {'green': 8, 'orange': 60, 'red': 7, 'total': 7

sample 4 --- is_lowercase_palindrome

  [aligned_p_plus] prefill='s == s'   | {'green': 14, 'orange': 8, 'red': 0, 'total': 22} | 14 green
  [aligned_p_plus] prefill='s[::-1]'  | {'green': 18, 'orange': 15, 'red': 6, 'total': 39} | 18 green
  [aligned_p_plus] prefill='s[:'      | {'green': 8, 'orange': 23, 'red': 3, 'total': 34} | 8 green
  [aligned_p_plus] prefill='s =='     | {'green': 8, 'orange': 7, 'red': 2, 'total': 17} | 8 green
  [aligned_p_minus] prefill='s == s.lower()' | {'green': 8, 'orange': 2, 'red': 2, 'total': 12} | 8 green
  [aligned_p_minus] prefill='s.lower()' | {'green': 28, 'orange': 28, 'red': 6, 'total': 62} | 28 green
  [aligned_p_minus] prefill='s.islower()' | {'green': 0, 'orange': 6, 'red': 6, 'total': 12} | 0 green
  [adversarial] prefill='not ('    | {'green': 186, 'orange': 92, 'red': 24, 'total': 302} | 186 green
  [adversarial] prefill='False if' | {'green': 34, 'orange': 16, 'red': 0, 'total': 50} | 34 green
  [adversarial] prefill='True if not' | {'green': 66, 'orange': 27, 'red': 1, 'tota

## Check a Tree to make sure it looks right

In [15]:
# SAMPLE_IDX: 0-based index into originals.
#   - target_predicate.samples[orig_indices[SAMPLE_IDX]] → the sample object
#   - tp_results.sample_results[SAMPLE_IDX]              → its MCTS results
SAMPLE_IDX = 4
rich_print(target_predicate.samples[orig_indices[SAMPLE_IDX]])

MultiPredicateSample(
    instruction='Write a Python function that returns True iff the string is all lowercase and is a 
palindrome.\n\n',
    code='def is_lowercase_palindrome(s: str) -> bool:\n    """Return True iff s consists entirely of lowercase 
characters and is a palindrome."""\n    return ',
    unit_tests=[
        ('aba', True),
        ('abc', False),
        ('ABA', False),
        ('Aba', False),
        ('racecar', True),
        ('hello', False),
        ('a', True),
        ('aA', False),
        ('abcba', True)
    ],
    prefills={
        'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
        'aligned_p_minus': ['s == s.lower()', 's.lower()', 's.islower()'],
        'adversarial': ['not (', 'False if', 'True if not'],
        'inconsequential': ['']
    },
    function_name='is_lowercase_palindrome',
    augmentation=None,
    p_minus=PredicateSpec(
        description='check whether s == s.lower() (all lowercase)',
        short_description='{var} consists entirely of lowercase characters',
        anchor_patterns=[
            "<=\\ 'Z'",
            '==\\ \\w+\\ and',
            '==\\s*\\w+\\.lower\\(\\)',
            'ABCDEFGHIJKLMNOPQRSTUVWXYZ',
            '\\.islower\\(\\)',
            '\\.lower\\(\\)',
            '\\ball\\b.*lower',
            '\\w+\\.islower\\(\\)',
            '\\w+\\.lower\\(\\)',
            "all\\(.*'a'\\s*<=",
            'char\\s*>=\\s*[\'"]a[\'"]',
            'islower',
            'isupper',
            'isupper\\(\\)',
            'lower\\(\\)'
        ]
    ),
    p_plus_short_description='{var} is a palindrome'
)

In [16]:
from src.goals.sampling import print_tree

PREFILL_IDX = -1

sr   = tp_results.sample_results[SAMPLE_IDX]
run  = sr.runs[PREFILL_IDX]
root = sr._trees.get(run.prefix)

if root is None:
    rich_print("[red]Tree missing — was this result saved without tree=root.to_dict()? Re-run MCTS.[/red]")
else:
    rich_print(
        f"[bold]Tree[/bold]  sample={SAMPLE_IDX}  fn=[cyan]{sr.sample.function_name}[/cyan]"
        f"  prefill={run.prefill!r}  eval={run.eval_counts}"
    )
    print_tree(root, show_colors=True)

Tree  sample=4  fn=is_lowercase_palindrome  prefill=''  eval={'green': 18, 'orange': 18, 'red': 1, 'total': 37}

[ROOT] visits=1713
    ├── ' s' (p=0.973, v=1595)
    │   ├── '.is' (p=0.832, v=1299)
    │   │   ├── 'lower' (p=1.000, v=1298)
    │   │   │   ├── '()' (p=1.000, v=1297)
    │   │   │   │   ├── ' and' (p=1.000, v=1296)
    │   │   │   │   │   ├── ' s' (p=1.000, v=1295)
    │   │   │   │   │   │   ├── ' ==' (p=1.000, v=1294)
    │   │   │   │   │   │   │   └── ' s' (p=1.000, v=1293)
    │   │   │   │   │   │   │       └── '[::-' (p=1.000, v=1292)
    │   │   │   │   │   │   │           └── '1' (p=1.000, v=1291)
    │   │   │   │   │   │   │               ├── ']\n' (p=0.777, v=1003) [TERMINAL] (' s.islower() and s == s[::-1]\n')
    │   │   │   │   │   │   │               └── ']\n\n' (p=0.223, v=287) [TERMINAL] (' s.islower() and s == s[::-1]\n\n')
    │   │   │   │   │   │   ├── '[::-' (p=0.000, v=0) (' s.islower() and s[::-')
    │   │   │   │   │   │   ├── '[:]' (p=0.000, v=0) (' s.islower() and s[:]')
    │   │   │   │   │   │   └── ' =' (p=0.000, v=0) (' s.islower() and s =')
    │   │   │   │   │   ├── ' all' (p=0.000, v=0) (' s.islower() and all')
    │   │   │   │   │   ├── ' (' (p=0.000, v=0) (' s.islower() and (')
    │   │   │   │   │   └── ' ' (p=0.000, v=0) (' s.islower() and ')
    │   │   │   │   ├── ' &' (p=0.000, v=0) (' s.islower() &')
    │   │   │   │   ├── ' ' (p=0.000, v=0) (' s.islower() ')
    │   │   │   │   └── ' &&' (p=0.000, v=0) (' s.islower() &&')
    │   │   │   ├── '(' (p=0.000, v=0) (' s.islower(')
    │   │   │   ├── '()`' (p=0.000, v=0) (' s.islower()`')
    │   │   │   └── '())' (p=0.000, v=0) (' s.islower())')
    │   │   ├── 'alpha' (p=0.000, v=0) (' s.isalpha')
    │   │   ├── 'low' (p=0.000, v=0) (' s.islow')
    │   │   └── '.lower' (p=0.000, v=0) (' s.is.lower')
    │   ├── ' ==' (p=0.164, v=256)
    │   │   ├── ' s' (p=1.000, v=224)
    │   │   │   ├── '[::-' (p=1.000, v=223)
    │   │   │   │   ├── '1' (p=1.000, v=222)
    │   │   │   │   │   ├── ']' (p=1.000, v=221)
    │   │   │   │   │   │   ├── ' and' (p=0.996, v=220)
    │   │   │   │   │   │   │   ├── ' s' (p=0.984, v=210)
    │   │   │   │   │   │   │   │   └── '.is' (p=0.996, v=209)
    │   │   │   │   │   │   │   │       └── 'lower' (p=1.000, v=208)
    │   │   │   │   │   │   │   │           ├── '()\n' (p=0.590, v=123) [TERMINAL] (' s == s[::-1] and s.islower()\n')
    │   │   │   │   │   │   │   │           └── '()\n\n' (p=0.406, v=84) [TERMINAL] (' s == s[::-1] and s.islower()\n\n')
    │   │   │   │   │   │   │   └── ' all' (p=0.016, v=9)
    │   │   │   │   │   │   │       ├── '(c' (p=0.797, v=9)
    │   │   │   │   │   │   │       │   └── '.is' (p=1.000, v=8)
    │   │   │   │   │   │   │       │       └── 'lower' (p=1.000, v=7)
    │   │   │   │   │   │   │       │           └── '()' (p=1.000, v=6)
    │   │   │   │   │   │   │       │               └── ' for' (p=1.000, v=5)
    │   │   │   │   │   │   │       │                   └── ' c' (p=1.000, v=4)
    │   │   │   │   │   │   │       │                       └── ' in' (p=1.000, v=3)
    │   │   │   │   │   │   │       │                           └── ' s' (p=1.000, v=2)
    │   │   │   │   │   │   │       │                               ├── ')\n' (p=0.621, v=1) [TERMINAL] (' s == s[::-1] and all(c.islower() for c in s)\n')
    │   │   │   │   │   │   │       │                               └── ')\n\n' (p=0.377, v=0) [TERMINAL] (' s == s[::-1] and all(c.islower() for c in s)\n\n')
    │   │   │   │   │   │   │       └── '(char' (p=0.201, v=0) (' s == s[::-1] and all(char')
    │   │   │   │   │   │   ├── ' if' (p=0.003, v=0) (' s == s[::-1] if')
    │   │   │   │   │   │   ├── ' ' (p=0.000, v=0) (' s == s[::-1] ')
    │   │   │   │   │   │   └── ' &' (p=0.000, v=0) (' s == s[::-1] &')
    │   │   │   │   │   ├── '])' (p=0.000, v=0) (' s == s[::-1])')
    │   │   │   │   │   ├── '].' (p=0.000, v=0) (' s == s[::-1].')
    │   │   │   │   │   └── ']\n' (p=0.000, v=0) [TERMINAL] (' s == s[::-1]\n')
    │   │   │   │   ├── '2' (p=0.000, v=0) (' s == s[::-2')
    │   │   │

In [17]:
tp_results.target_predicate_name, sr.sample.p_minus.description, sr.runs[PREFILL_IDX].prefill

('is_palindrome', 'check whether s == s.lower() (all lowercase)', '')

In [18]:
from rich.table import Table


def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict

    groups = defaultdict(list)

    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)

    walk(root)
    return dict(groups)


groups = get_green_solutions_by_first_token(root, prefix="")

rich_print(
    f"[bright_cyan]p+={tp_results.target_predicate_name}[/bright_cyan], [bright_magenta]p-={sr.sample.p_minus.description if sr.sample.p_minus else '?'}[/bright_magenta]  fn={sr.sample.function_name}  prefill={run.prefill!r}  eval={run.eval_counts}"
)

table = Table(title="Green solutions grouped by first token", show_lines=True)
table.add_column("First token", style="cyan", no_wrap=True)
table.add_column("Count", justify="right", style="bold")
table.add_column("Green completions", overflow="fold")

for first_tok, codes in sorted(
    groups.items(), key=lambda kv: (-len(kv[1]), repr(kv[0]))
):
    completions = "\n".join(f"{i + 1}. {code!r}" for i, code in enumerate(codes))
    table.add_row(repr(first_tok), str(len(codes)), completions if completions else "—")

rich_print(table)

p+=is_palindrome, p-=check whether s == s.lower() (all lowercase)  fn=is_lowercase_palindrome  prefill=''  
eval={'green': 18, 'orange': 18, 'red': 1, 'total': 37}

                                 Green solutions grouped by first token                                 
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ First token ┃ Count ┃ Green completions                                                              ┃
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ' all'      │    10 │ 1. ' all(c.islower() for c in s) and s == s[::-1]\n'                           │
│             │       │ 2. ' all(c.islower() for c in s) and s == s[::-1]\n\n'                         │
│             │       │ 3. ' all(char.islower() for char in s) and s == s[::-1]\n'                     │
│             │       │ 4. ' all(char.islower() for char in s) and s == s[::-1]\n\n'                   │
│             │       │ 5. ' all(char.isdigit() or char.islower() for char in s) and s == s[::-1]\n'   │
│             │       │ 6. ' all(char.isdigit() or char.islower() for char in s) and s == s[::-1]\n\n' │
│             │       │ 7. ' all(char == char.lower() for char in s) and s == s[::-1]\n'               │
│             │       │ 8. ' all(char == char.lower() for char in s) and s == s[::-1]\n\n'             │
│             │       │ 9. ' all(char.lower() == char for char in s) and s == s[::-1]\n'               │
│             │       │ 10. ' all(char.lower() == char for char in s) and s == s[::-1]\n\n'            │
├─────────────┼───────┼────────────────────────────────────────────────────────────────────────────────┤
│ ' s'        │     6 │ 1. ' s == s[::-1] and all(c.islower() for c in s)\n'                           │
│             │       │ 2. ' s == s[::-1] and all(c.islower() for c in s)\n\n'                         │
│             │       │ 3. ' s.lower() == s and s == s[::-1]\n'                                        │
│             │       │ 4. ' s.lower() == s and s == s[::-1]\n\n'                                      │
│             │       │ 5. ' s.lower()==s and s == s[::-1]\n'                                          │
│             │       │ 6. ' s.lower()==s and s == s[::-1]\n\n'                                        │
├─────────────┼───────┼────────────────────────────────────────────────────────────────────────────────┤
│ '0'         │     2 │ 1. '0 == len(s) or (s.islower() and s == s[::-1])\n\n'                         │
│             │       │ 2. '0 == len(s) or (s.islower() and s == s[::-1])\n'                           │
└─────────────┴───────┴────────────────────────────────────────────────────────────────────────────────┘

## Anchor Extration with Oracle LLMs

In [19]:
from src.goals.anchor import extract_anchor_tokens

# ── Select which samples to run oracle extraction on ───────────────────────
SAMPLE_INDICES = [SAMPLE_IDX]   # e.g. [0, 2] for multiple

completions_selected = [
    tp_results.completions_per_sample[i] if i in SAMPLE_INDICES else {}
    for i in range(len(tp_results.completions_per_sample))
]

p_plus_anchors, p_minus_anchors_per_sample = extract_anchor_tokens(
    tp=target_predicate,
    completions_per_sample=completions_selected,
    use_cache=True,
)

# ── Inspect ────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]P+ anchors[/bold] ([cyan]{target_predicate.p_plus.description}[/cyan]):")
rich_print(p_plus_anchors)

for i in SAMPLE_INDICES:
    sample = target_predicate.samples[i]
    p_minus_desc = sample.p_minus.description if sample.p_minus else "?"
    rich_print(f"\n[bold]P- anchors  sample {i}[/bold] ([magenta]{p_minus_desc}[/magenta]):")
    rich_print(p_minus_anchors_per_sample[i])

2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4: both predicates already identified in completion — skipping oracle call.
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4: both predicates already identified in completion — skipping oracle call.
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4: both predicates already identified in completion — skipping oracle call.
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4: both predicates already identified in completion — skipping oracle call.
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4: both predicates already identified in completion — skipping oracle call.
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4: both predicates already identified in completion — skipping oracle call.
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4: both predicates already identified in completion — skippin

2026-03-16 19:58:01 src.utils.oracle_llms DEBUG    found cached claude response for 4f04acf5585e45332767d46732f0107f - loading
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4 prefix='def is_lowercase_palindrome(s:': p+=['\\w+\\[::-1\\]', '==\\s*\\w+\\[::-1\\]']  p-=['\\w+\\.islower\\(\\)', '==\\s*\\w+\\.lower\\(\\)', '\\.lower\\(\\)']
2026-03-16 19:58:01 src.utils.oracle_llms DEBUG    found cached claude response for 4f04acf5585e45332767d46732f0107f - loading
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4 prefix='def is_lowercase_palindrome(s:': p+=['\\w+\\[::-1\\]', '==\\s*\\w+\\[::-1\\]']  p-=['\\w+\\.islower\\(\\)', '==\\s*\\w+\\.lower\\(\\)', '\\.lower\\(\\)']
2026-03-16 19:58:01 src.utils.oracle_llms DEBUG    found cached claude response for 38cfc7ab6bad680cd5873a17f74c6520 - loading
2026-03-16 19:58:01 src.goals.anchor DEBUG    [is_palindrome] sample=4 prefix='def is_lowercase_palindrome(s:': p+=['\\w+\\[::-1\\]', '==\\s*\\w+\\[::-1\

P+ anchors (check whether s is a palindrome (s == s[::-1])):

[
    '::\\-1\\]',
    '==\\ \\w+',
    '==\\ \\w+\\[',
    '==\\ \\w+\\[::\\-1\\]',
    '==\\s*\\w+\\[::-1\\]',
    '==\\w+',
    '\\[::-1\\]',
    '\\[::\\-1\\]',
    '\\w+\\ !=\\ \\w+',
    '\\w+\\ !=\\ \\w+\\[',
    '\\w+\\ !=\\ \\w+\\[::\\-1\\]',
    '\\w+\\ ==',
    '\\w+\\ ==\\ \\w+',
    '\\w+\\ ==\\ \\w+\\[',
    '\\w+\\ ==\\ \\w+\\[::\\-1\\]',
    '\\w+\\.lower\\(\\)\\ ==',
    '\\w+\\.lower\\(\\)\\[::\\-1\\]',
    '\\w+\\[::-1\\]',
    '\\w+\\[::\\-',
    '\\w+\\[::\\-1\\]',
    '\\w+\\[::\\-\\ 1\\]',
    '\\w+\\[\\-i\\-1\\]',
    'enumerate\\(\\w+\\)',
    'lower\\(\\)\\[::\\-1\\]',
    'reversed',
    'reversed\\(\\w+\\)'
]

P- anchors  sample 4 (check whether s == s.lower() (all lowercase)):

[
    "<=\\ 'Z'",
    '==\\ \\w+\\ and',
    '==\\s*\\w+\\.lower\\(\\)',
    'ABCDEFGHIJKLMNOPQRSTUVWXYZ',
    '\\.islower\\(\\)',
    '\\.lower\\(\\)',
    '\\ball\\b.*lower',
    '\\w+\\.islower\\(\\)',
    '\\w+\\.lower\\(\\)',
    "all\\(.*'a'\\s*<=",
    'char\\s*>=\\s*[\'"]a[\'"]',
    'islower',
    'isupper',
    'isupper\\(\\)',
    'lower\\(\\)'
]

In [20]:
# ── Set anchors on target_predicate and save to disk ──────────────────────
# Review the printed anchors above, edit the lists if needed, then run this cell.

target_predicate.p_plus.anchor_patterns = p_plus_anchors

for i in SAMPLE_INDICES:
    sample = target_predicate.samples[i]
    if sample.p_minus and p_minus_anchors_per_sample[i]:
        sample.p_minus.anchor_patterns = p_minus_anchors_per_sample[i]

target_predicate.save()   # overwrites data/multi_predicate/<name>.json

rich_print(f"[green]Saved updated anchor tokens → {target_predicate._source_path}[/green]")

2026-03-16 19:58:03 src.goals.predicate_data INFO     Saved TargetPredicate 'is_palindrome' to /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json


Saved updated anchor tokens → /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json

## Checking predicate labelling with anchor tokens

In [24]:
from src.goals.anchor import (
    label_predicate_spans,
    get_labeled_leaves,
    show_span_annotation,
    show_clause_spans,
    extract_predicate_clauses,
    print_labeled_tree,
)
from rich.console import Console
from rich.table import Table
from rich import box

_console = Console(highlight=False, width=200)

# ── Select which run to inspect ─────────────────────────────────────────────
sample_idx = SAMPLE_IDX
run_idx    = PREFILL_IDX

sample_result = tp_results.sample_results[sample_idx]
run           = sample_result.runs[run_idx]

if run.prefix not in sample_result._trees:
    raise RuntimeError(f"Tree not in memory for prefix {run.prefix!r} — re-run MCTS or load from JSON.")

root            = sample_result._trees[run.prefix]
p_plus_anchors  = target_predicate.p_plus.anchor_patterns
p_minus_anchors = target_predicate.samples[sample_idx].p_minus.anchor_patterns

_console.print(f"P+ anchors : {p_plus_anchors}")
_console.print(f"P- anchors : {p_minus_anchors}")
_console.print(f"Prefill    : {run.prefill!r}")

# run.prefix = sample.code + prefill.  The MCTS tree includes the prefill as
# its first forced tokens, so path_text already starts with the prefill text.
# Using run.prefix directly would double-count it in full_text and corrupt
# clause extraction.  Strip the prefill to get the bare code stub.
code_prefix = run.prefix[:-len(run.prefill)] if run.prefill else run.prefix

# ── Label spans ─────────────────────────────────────────────────────────────
labels = label_predicate_spans(root, p_plus_anchors, p_minus_anchors, prefix=code_prefix)
leaves = get_labeled_leaves(root, labels)

# ─────────────────────────────────────────────────────────────────────────────
# View 1: Leaf table — anchor-level (which predicate comes first)
# ─────────────────────────────────────────────────────────────────────────────
_LABEL_COLOR = {
    "p_plus_first":  "bold bright_cyan",
    "p_minus_first": "bold bright_magenta",
    "mixed":         "bold bright_yellow",
    "unlabeled":     "dim white",
}

leaf_table = Table(box=box.SIMPLE_HEAVY, show_lines=False, min_width=120)
leaf_table.add_column("#",       style="dim",  width=4,  no_wrap=True)
leaf_table.add_column("label",                 width=14, no_wrap=True)
leaf_table.add_column("P+ clause (depth)",     width=30, no_wrap=True)
leaf_table.add_column("P- clause (depth)",     width=30, no_wrap=True)
leaf_table.add_column("path text", overflow="fold")

for i, (leaf, path_text, lbl) in enumerate(leaves):
    pl = labels.get(leaf)
    p_plus_clause, p_minus_clause = extract_predicate_clauses(
        leaf, code_prefix, p_plus_anchors, p_minus_anchors
    )
    style = _LABEL_COLOR.get(lbl, "")
    def _fmt(clause):
        if clause is None: return "—"
        return f"{clause.text!r} (d {clause.start_depth}→{clause.end_depth})"
    leaf_table.add_row(
        str(i),
        f"[{style}]{lbl}[/{style}]",
        _fmt(p_plus_clause),
        _fmt(p_minus_clause),
        repr(path_text),
    )

n_both = sum(1 for _, _, lbl in leaves if lbl in ("p_plus_first", "p_minus_first"))
_console.print(f"\n[bold]View 1 — Green leaf labels[/bold]  "
               f"({len(leaves)} green leaves, {n_both} with both spans identified)")
_console.print(leaf_table)

# ─────────────────────────────────────────────────────────────────────────────
# View 2: Labeled tree (mixed nodes = contrastive branching points)
# ─────────────────────────────────────────────────────────────────────────────
_console.print("\n[bold]View 2 — Labeled tree  [bright_yellow](±)[/bright_yellow] = mixed / contrastive branching point[/bold]")
print_labeled_tree(root, labels, max_depth=8)

# ─────────────────────────────────────────────────────────────────────────────
# View 3: Per-leaf clause span annotations
# Shows full clause text + depth range for P+ and P- within each branch.
# Change SHOW_FILTER to narrow down: "all", "unlabeled", "p_plus_first", "p_minus_first"
# ─────────────────────────────────────────────────────────────────────────────
_console.print("\n[bold]View 3 — Per-leaf clause spans[/bold]")
SHOW_FILTER = "all"
shown = 0
for leaf, path_text, lbl in leaves:
    if SHOW_FILTER != "all" and lbl != SHOW_FILTER:
        continue
    style = _LABEL_COLOR.get(lbl, "")
    _console.print(f"  [dim]leaf {shown}[/dim]  [{style}]{lbl}[/{style}]")
    _console.print(show_clause_spans(leaf, code_prefix, p_plus_anchors, p_minus_anchors))
    _console.print()
    shown += 1
    if shown >= 8:
        _console.print(f"  [dim]... {len(leaves) - shown} more leaves not shown (increase cap or set SHOW_FILTER)[/dim]")
        break

P+ anchors : ['::\\-1\\]', '==\\ \\w+', '==\\ \\w+\[', '==\\ \\w+\[::\\-1\\]', '==\\s*\\w+\[::-1\\]', '==\\w+', '\[::-1\\]', '\[::\\-1\\]', '\\w+\\ !=\\ \\w+', '\\w+\\ !=\\ \\w+\[', '\\w+\\ !=\\ 
\\w+\[::\\-1\\]', '\\w+\\ ==', '\\w+\\ ==\\ \\w+', '\\w+\\ ==\\ \\w+\[', '\\w+\\ ==\\ \\w+\[::\\-1\\]', '\\w+\\.lower\\(\\)\\ ==', '\\w+\\.lower\\(\\)\[::\\-1\\]', '\\w+\[::-1\\]', '\\w+\[::\\-', 
'\\w+\[::\\-1\\]', '\\w+\[::\\-\\ 1\\]', '\\w+\[\\-i\\-1\\]', 'enumerate\\(\\w+\\)', 'lower\\(\\)\[::\\-1\\]', 'reversed', 'reversed\\(\\w+\\)']

P- anchors : ["<=\\ 'Z'", '==\\ \\w+\\ and', '==\\s*\\w+\\.lower\\(\\)', 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', '\\.islower\\(\\)', '\\.lower\\(\\)', '\\ball\\b.*lower', '\\w+\\.islower\\(\\)', 
'\\w+\\.lower\\(\\)', "all\\(.*'a'\\s*<=", 'char\\s*>=\\s*[\'"]a[\'"]', 'islower', 'isupper', 'isupper\\(\\)', 'lower\\(\\)']

Prefill    : ''

2026-03-16 21:08:59 src.goals.anchor DEBUG    label_predicate_spans: 18 green leaves — both=18, p+_only=0, p-_only=0, unlabeled=0


View 1 — Green leaf labels  (18 green leaves, 18 with both spans identified)

 #      label            P+ clause (depth)                P- clause (depth)                path text                                                                    
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  0      p_minus_first    's == s[::-1]' (d 12→17)         'all(c.islower() for c in s)'…   ' all(c.islower() for c in s) and s == s[::-1]\n'                            
  1      p_minus_first    's == s[::-1]' (d 12→17)         'all(c.islower() for c in s)'…   ' all(c.islower() for c in s) and s == s[::-1]\n\n'                          
  2      p_minus_first    's == s[::-1]' (d 13→18)         'all(char == char.lower() for…   ' all(char == char.lower() for char in s) and s == s[::-1]\n'                
  3      p_minus_first    's == s[::-1]' (d 13→18)         'all(char == char.lower() for…   ' all(char == char.lower() for char in s) and s == s[::-1]\n\n'              
  4      p_minus_first    's == s[::-1]' (d 16→21)         'all(char.isdigit() or char.i…   ' all(char.isdigit() or char.islower() for char in s) and s == s[::-1]\n'    
  5      p_minus_first    's == s[::-1]' (d 16→21)         'all(char.isdigit() or char.i…   ' all(char.isdigit() or char.islower() for char in s) and s == s[::-1]\n\n'  
  6      p_minus_first    's == s[::-1]' (d 12→17)         'all(char.islower() for char …   ' all(char.islower() for char in s) and s == s[::-1]\n'                      
  7      p_minus_first    's == s[::-1]' (d 12→17)         'all(char.islower() for char …   ' all(char.islower() for char in s) and s == s[::-1]\n\n'                    
  8      p_minus_first    'all(char.lower() == char for…   'all(char.lower() == char for…   ' all(char.lower() == char for char in s) and s == s[::-1]\n'                
  9      p_minus_first    'all(char.lower() == char for…   'all(char.lower() == char for…   ' all(char.lower() == char for char in s) and s == s[::-1]\n\n'              
  10     p_minus_first    's.lower()==s' (d 1→5)           's.lower()==s' (d 1→5)           ' s.lower()==s and s == s[::-1]\n'                                           
  11     p_minus_first    's.lower()==s' (d 1→5)           's.lower()==s' (d 1→5)           ' s.lower()==s and s == s[::-1]\n\n'                                         
  12     p_plus_first     's == s[::-1]' (d 1→7)           'all(c.islower() for c in s)'…   ' s == s[::-1] and all(c.islower() for c in s)\n'                            
  13     p_plus_first     's == s[::-1]' (d 1→7)           'all(c.islower() for c in s)'…   ' s == s[::-1] and all(c.islower() for c in s)\n\n'                          
  14     p_plus_first     's.lower() == s' (d 1→6)         's.lower() == s' (d 1→6)         ' s.lower() == s and s == s[::-1]\n'                                         
  15     p_plus_first     's.lower() == s' (d 1→6)         's.lower() == s' (d 1→6)         ' s.lower() == s and s == s[::-1]\n\n'                                       
  16     p_plus_first     '(s.islower() and s == s[::-1…   '(s.islower() and s == s[::-1…   '0 == len(s) or (s.islower() and s == s[::-1])\n'                            
  17     p_plus_first     '(s.islower() and s == s[::-1…   '(s.islower() and s == s[::-1…   '0 == len(s) or (s.islower() and s == s[::-1])\n\n'

... (16 outputs left out)

Path: ' all(char.islower() for char in s) and s == s[::-1]
'
  All clauses: ['all(char.islower() for char in s)', 's == s[::-1]']
  P+: 's == s[::-1]'                            depth 12 → 17
  P-: 'all(char.islower() for char in s)'       depth 1 → 11

leaf 7  p_minus_first

Path: ' all(char.islower() for char in s) and s == s[::-1]

'
  All clauses: ['all(char.islower() for char in s)', 's == s[::-1]']
  P+: 's == s[::-1]'                            depth 12 → 17
  P-: 'all(char.islower() for char in s)'       depth 1 → 11

... 10 more leaves not shown (increase cap or set SHOW_FILTER)

In [23]:
import re
                                                                                                                                                                                                                                                     
# Paste the two clauses directly                                                                                                                                                                                                                 
clauses = [                                                                                                                                                                                                                                        
    'all(char == char.lower() for char in s)',  # clause 1
    's == s[::-1]',                              # clause 2
]

p_plus_anchors  = target_predicate.p_plus.anchor_patterns
p_minus_anchors = target_predicate.samples[sample_idx].p_minus.anchor_patterns

for label, anchors in [("P+", p_plus_anchors), ("P-", p_minus_anchors)]:
    print(f"\n── {label} ──")
    for anchor in anchors:
        hits = []
        for i, clause in enumerate(clauses):
            try:
                if re.search(anchor, clause):
                    hits.append(i)
            except re.error:
                hits.append(f"INVALID:{anchor}")
        if hits:
            print(f"  {anchor!r:50s}  → clause(s) {hits}")


── P+ ──
  '::\\-1\\]'                                         → clause(s) [1]
  '==\\ \\w+'                                         → clause(s) [0, 1]
  '==\\ \\w+\\['                                      → clause(s) [1]
  '==\\ \\w+\\[::\\-1\\]'                             → clause(s) [1]
  '==\\s*\\w+\\[::-1\\]'                              → clause(s) [1]
  '\\[::-1\\]'                                        → clause(s) [1]
  '\\[::\\-1\\]'                                      → clause(s) [1]
  '\\w+\\ =='                                         → clause(s) [0, 1]
  '\\w+\\ ==\\ \\w+'                                  → clause(s) [0, 1]
  '\\w+\\ ==\\ \\w+\\['                               → clause(s) [1]
  '\\w+\\ ==\\ \\w+\\[::\\-1\\]'                      → clause(s) [1]
  '\\w+\\[::-1\\]'                                    → clause(s) [1]
  '\\w+\\[::\\-'                                      → clause(s) [1]
  '\\w+\\[::\\-1\\]'                                  → clause(s) [1]



## Checking contrasting sampling

In [31]:
from src.goals.predicate_probe import collect_labeled_nodes
MIN_P_GREEN = 0.5

labeled_i = collect_labeled_nodes(
    tp_results=tp_results,
    tp = target_predicate,
    mt = mt,
    sample_indices=[SAMPLE_IDX], 
    min_p_green=MIN_P_GREEN
)
labeled_i

2026-03-03 13:18:30 src.goals.predicate_probe DEBUG    Sample 4, prefill 's == s': 14 green leaves → 96 labeled nodes (52 dropped by p_green < 0.5).
2026-03-03 13:18:30 src.goals.predicate_probe DEBUG    Sample 4, prefill 's[::-1]': 18 green leaves → 136 labeled nodes (68 dropped by p_green < 0.5).
2026-03-03 13:18:30 src.goals.predicate_probe DEBUG    Sample 4, prefill 's[:': 8 green leaves → 12 labeled nodes (10 dropped by p_green < 0.5).
2026-03-03 13:18:30 src.goals.predicate_probe DEBUG    Sample 4, prefill 's ==': 8 green leaves → 50 labeled nodes (44 dropped by p_green < 0.5).
2026-03-03 13:18:30 src.goals.predicate_probe DEBUG    Sample 4, prefill 's == s.lower()': 8 green leaves → 0 labeled nodes (0 dropped by p_green < 0.5).
2026-03-03 13:18:30 src.goals.predicate_probe DEBUG    Sample 4, prefill 's.lower()': 28 green leaves → 16 labeled nodes (0 dropped by p_green < 0.5).
2026-03-03 13:18:30 src.goals.predicate_probe DEBUG    Sample 4, prefill 's.islower()': 0 green leaves →

2026-03-03 13:18:31 src.goals.predicate_probe DEBUG    Sample 4, prefill 'not (': 186 green leaves → 2610 labeled nodes (162 dropped by p_green < 0.5).
2026-03-03 13:18:31 src.goals.predicate_probe DEBUG    Sample 4, prefill 'False if': 34 green leaves → 366 labeled nodes (8 dropped by p_green < 0.5).
2026-03-03 13:18:31 src.goals.predicate_probe DEBUG    Sample 4, prefill 'True if not': 66 green leaves → 250 labeled nodes (34 dropped by p_green < 0.5).
2026-03-03 13:18:31 src.goals.predicate_probe DEBUG    Sample 4, prefill '': 18 green leaves → 190 labeled nodes (14 dropped by p_green < 0.5).
2026-03-03 13:18:31 src.goals.predicate_probe INFO     collect_labeled_nodes: 3726 total labeled nodes across 9 trees.


[LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=108, depth=6, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=109, depth=7, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=110, depth=8, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=111, depth=9, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=108, depth=6, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=109, depth=7, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=110, depth=8, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=111, depth=9, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=1.00, abs_pos=108, depth=6, tree='""\n    return s == s'),
 LabeledNode('p_minus', win='second', p_green=1.00, abs

In [32]:
ln = labeled_i[4]
print(ln)
rich_print(f"[white]{ln.tree_id}[/white][bright_green]{ln.node.get_path_text()}[/bright_green]")

LabeledNode('p_minus', win='second', p_green=0.93, abs_pos=108, depth=6, tree='""\n    return s == s')


def is_lowercase_palindrome(s: str) -> bool:
    """Return True iff s consists entirely of lowercase characters and is a palindrome."""
    return s == s[::-1] and s.lower

In [ ]:
from src.goals.predicate_probe import (
    collect_labeled_nodes,
    sample_multipred_pairs,
    print_node_label_summary,
    print_pairs_sample,
    pairs_position_histogram,
)

MIN_P_GREEN = 0.5

all_labeled = []
all_pairs_by_pos = {}

# Iterate over sample_results (one entry per original — augmented samples
# share unit tests so no separate MCTS results exist for them).
for sample_idx in range(len(tp_results.sample_results)):
    labeled_i = collect_labeled_nodes(
        tp_results, target_predicate, mt,
        sample_indices=[sample_idx], min_p_green=MIN_P_GREEN,
    )
    all_labeled.extend(labeled_i)
    if not labeled_i:
        continue
    pairs_i = sample_multipred_pairs(labeled_i, allow_same_tree=False, abs_pos_tolerance=0)
    for pos, pairs in pairs_i.items():
        all_pairs_by_pos.setdefault(pos, []).extend(pairs)

print_node_label_summary(all_labeled)
pairs_position_histogram(all_pairs_by_pos)
print_pairs_sample(all_pairs_by_pos, n=6)

In [34]:
# Inspect outcome probabilities at a specific depth
from src.goals.probe import collect_nodes_at_depth, compute_outcome_probs

depth_to_inspect = 3

nodes = collect_nodes_at_depth(root, depth_to_inspect)
print(f"Nodes at depth {depth_to_inspect}: {len(nodes)}")

for node in nodes[:10]:  # Show first 10
    if not node.is_leaf() and not node.is_terminal:
        probs = compute_outcome_probs(node)
        path_text = node.get_path_text()[:40]
        print(f"  '{path_text}...' -> {probs}")

Nodes at depth 3: 28
  ' s.islower...' -> OutcomeProbs(green=0.000, red=1.000, uncertain=0.000)
  ' s == s...' -> OutcomeProbs(green=0.013, red=0.980, uncertain=0.007)
  ' s == ''....' -> OutcomeProbs(green=0.000, red=0.947, uncertain=0.053)
  ' s == ""....' -> OutcomeProbs(green=0.000, red=0.925, uncertain=0.075)
  ' s == ...' -> OutcomeProbs(green=0.000, red=0.905, uncertain=0.095)
  ' s.lower()...' -> OutcomeProbs(green=0.909, red=0.000, uncertain=0.091)
  ' s.lower()....' -> OutcomeProbs(green=0.000, red=0.966, uncertain=0.034)


In [ ]:
# Look at a specific contrastive pair in detail
if all_pairs_by_pos:
    first_abs_pos = max(all_pairs_by_pos.keys())
    pairs = all_pairs_by_pos[first_abs_pos]

    if pairs:
        pair = pairs[0]
        rich_print(
            f"[bold]Example Contrastive Pair at abs_position={first_abs_pos}:[/bold]"
        )
        rich_print("[bright_cyan]Positive (p_plus):[/bright_cyan]")
        print(
            f"  prefill_type={pair.positive.prefill_type!r}  prefill={pair.positive.prefill!r}"
        )
        print(
            f"  label={pair.positive.label!r}  window={pair.positive.window!r}  depth={pair.positive.depth}  p_green={pair.positive.p_green:.2f}"
        )
        print(
            f"  outcome_probs={compute_outcome_probs(pair.positive.node)}  n_children={len(pair.positive.node.children)}"
        )
        prefill_idx = pair.positive.tree_id.index(pair.positive.prefill)
        rich_print(
            f"[white]{pair.positive.tree_id[:prefill_idx]}[/white][bright_yellow]{pair.positive.prefill}[/bright_yellow][bright_green]{pair.positive.node.get_path_text()}[/bright_green]"
        )

        print()
        rich_print("[bright_magenta]Negative (p_minus):[/bright_magenta]")
        print(
            f"  prefill_type={pair.negative.prefill_type!r}  prefill={pair.negative.prefill!r}"
        )
        print(
            f"  label={pair.negative.label!r}  window={pair.negative.window!r}  depth={pair.negative.depth}  p_green={pair.negative.p_green:.2f}"
        )
        print(
            f"  outcome_probs={compute_outcome_probs(pair.negative.node)}  n_children={len(pair.negative.node.children)}"
        )
        prefill_idx = pair.negative.tree_id.index(pair.negative.prefill)
        rich_print(
            f"[white]{pair.negative.tree_id[:prefill_idx]}[/white][bright_yellow]{pair.negative.prefill}[/bright_yellow][bright_green]{pair.negative.node.get_path_text()}[/bright_green]"
        )
else:
    print(
        "[yellow]No contrastive pairs found. Try adjusting thresholds or running more MCTS iterations.[/yellow]"
    )

Example Contrastive Pair at abs_position=132:

Positive (p_plus):

  prefill_type='aligned_p_plus'  prefill='s[:'
  label='p_plus'  window='second'  depth=21  p_green=0.89
  outcome_probs=OutcomeProbs(green=0.889, red=0.000, uncertain=0.111)  n_children=3


def is_upper_palindrome(s: str) -> bool:
    """Return True iff s is non-empty, starts with an uppercase letter, and is a palindrome."""
    return s[:1] >= 'A' and s[:1] <= 'Z' and s == s[::-1

Negative (p_minus):

  prefill_type='adversarial'  prefill='not ('
  label='p_minus'  window='second'  depth=21  p_green=1.00
  outcome_probs=OutcomeProbs(green=1.000, red=0.000, uncertain=0.000)  n_children=2


def is_upper_palindrome(s: str) -> bool:
    """Return True iff s is non-empty, starts with an uppercase letter, and is a palindrome."""
    return not (len(s) == 0) and s == s[::-1] and s[0].isupper